In [2]:
import torch

import triton
import triton.language as tl
from einops import einsum, rearrange

DEVICE = torch.device("cuda:0")

In [12]:
#attn
def _attention_and_lse(q, k, v, is_causal=True):
    n_queries = q.shape[-2]
    n_keys = k.shape[-2]
    d = q.shape[-1]
    scale = 1 / (d ** 0.5)
    S = einsum(q, k, '... q d, ... k d -> ... q k') * scale
    if is_causal:
        S = torch.where(
            torch.arange(n_queries, device=S.device)[None, :, None] >= torch.arange(n_keys, device=S.device)[None, None, :],
            S,
            -1e6
        )
    P = torch.softmax(S, dim=-1)
    o = einsum(P, v, '... q k, ... k d -> ... q d')
    L = torch.logsumexp(S, dim=-1)
    return o, L

In [7]:
def _make_attn_inputs(n_queries=128, D=64, dtype=torch.float32, device=torch.device("cuda:0"), batch_size=1):
    torch.random.manual_seed(0)
    n_keys = n_queries
    q = torch.randn(batch_size, n_queries, D, dtype=dtype, device=device, requires_grad=True)
    k = torch.randn(batch_size, n_keys, D, dtype=dtype, device=device, requires_grad=True)
    v = torch.randn(batch_size, n_keys, D, dtype=dtype, device=device, requires_grad=True)
    do = torch.randn(batch_size, n_queries, D, dtype=dtype, device=device)

    return q, k, v, do

In [9]:
from cs336_systems.flash_attn import FlashAttnPytorch
#my_flash_pytorch
flash_pytorch = torch.compile(FlashAttnPytorch.apply)
def flashattention_pytorch_forward(q, k, v, is_causal):
    return flash_pytorch(q, k, v, is_causal)

q, k, v, do = _make_attn_inputs()
o = flashattention_pytorch_forward(q, k, v, True)
print(o.shape)

torch.Size([1, 128, 64])


In [10]:
from cs336_systems.flash_attn_triton import FlashAttnTriton
#my_flash_triton
flash_triton = torch.compile(FlashAttnTriton.apply)
def flashattention_triton_forward(q, k, v, is_causal):
    return flash_triton(q, k, v, is_causal)

q, k, v, do = _make_attn_inputs()
o = flashattention_triton_forward(q, k, v, True)
print(o.shape)


torch.Size([1, 128, 64])


I can learn from https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html#sphx-glr-getting-started-tutorials-03-matrix-multiplication-py

In [18]:

#@triton.testing.perf_report(configs)

def benchmark(seq_len, D, dtype, provider):
    q, k, v, do = _make_attn_inputs(n_queries=seq_len, D=D, dtype=dtype)
    quantiles = [0.5, 0.2, 0.8]
    if provider == "attn":
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: _attention_and_lse(q, k, v), quantiles=quantiles)
    elif provider == "my_flash_pytorch":
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: flashattention_pytorch_forward(q, k, v, True), quantiles=quantiles)
    elif provider == "my_flash_triton":
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: flashattention_triton_forward(q, k, v, True), quantiles=quantiles)
    #attn 的flops  6×B×L×D² + 4×B×L²×D
    perf = lambda ms: (6 * seq_len * D * D + 4 * seq_len * seq_len * D ) * 1e-12 / (ms * 1e-3) # flops
    return perf(ms), perf(max_ms), perf(min_ms)


#benchmark.run(show_plots=True, print_data=True)

In [19]:
for dt in [torch.float32, torch.bfloat16]:
    for D in [2**i for i in range(4, 8)]:
        #for seq_len in [2**i for i in range(7, 17)]:
        for seq_len in [2**i for i in range(7, 11)]:
            for provider in ["attn", "my_flash_pytorch", "my_flash_triton"]:
                ms, min_ms, max_ms = benchmark(seq_len, D, dt, provider)
                print(dt, D, seq_len, provider, ms)

torch.float32 16 128 attn 0.031179488257994274
torch.float32 16 128 my_flash_pytorch 0.0009910350207898593
torch.float32 16 128 my_flash_triton 0.10133332974908679
torch.float32 16 256 attn 0.12108107580624441
torch.float32 16 256 my_flash_pytorch 0.0012538483188871464
torch.float32 16 256 my_flash_triton 0.2986666742116215
torch.float32 16 512 attn 0.3898181787735421
torch.float32 16 512 my_flash_pytorch 0.0014003883718835884
torch.float32 16 512 my_flash_triton 0.7146666413882964
torch.float32 16 1024 attn 0.8721268202630582
torch.float32 16 1024 my_flash_pytorch 0.001544830971677202
torch.float32 16 1024 my_flash_triton 1.635902465835205
torch.float32 32 128 attn 0.080457140708213
torch.float32 32 128 my_flash_pytorch 0.0023273329282493607


loc("/tmp/torchinductor_root/kk/ckk2kdlnfk6spibnmnrcvv543e4lyuzeswfdq6h3mnbzrxmunei3.py":106:11): error: operation scheduled before its operands


torch.float32 32 128 my_flash_triton 0.20114285438393759
torch.float32 32 256 attn 0.2560000113688022
torch.float32 32 256 my_flash_pytorch 0.002711259694575779
torch.float32 32 256 my_flash_triton 0.50782380812006
torch.float32 32 512 attn 0.8334883719937359
torch.float32 32 512 my_flash_pytorch 0.0030191222762901712
torch.float32 32 512 my_flash_triton 1.086060597578028
torch.float32 32 1024 attn 1.6940247265442616
torch.float32 32 1024 my_flash_pytorch 0.003169733927904155
torch.float32 32 1024 my_flash_triton 2.3657930445000632
torch.float32 64 128 attn 0.20479999452999673
torch.float32 64 128 my_flash_pytorch 0.005808752182948287
torch.float32 64 128 my_flash_triton 0.44800000113109123
torch.float32 64 256 attn 0.5776410456217886
torch.float32 64 256 my_flash_pytorch 0.006331646843187452


loc("/tmp/torchinductor_root/nt/cntthznzf5fbc36ztx67vz2q23h3ar5d5wg3i76a7d4a6nufuptl.py":106:11): error: operation scheduled before its operands


torch.float32 64 256 my_flash_triton 0.8664615482264723
torch.float32 64 512 attn 1.655829739346555
torch.float32 64 512 my_flash_pytorch 0.006672439536574981
torch.float32 64 512 my_flash_triton 1.7294221960044205
torch.float32 64 1024 attn 3.1857777294818277
torch.float32 64 1024 my_flash_pytorch 0.006514512650735756
torch.float32 64 1024 my_flash_triton 3.4965854231592175
torch.float32 128 128 attn 0.5535134893999744
torch.float32 128 128 my_flash_pytorch 0.016820492141834485


loc("/tmp/torchinductor_root/qk/cqksoi5f3bx2oowkiacenhwasozzxm5cynqgy3sq7ilj4tpg6r3f.py":106:11): error: operation scheduled before its operands


BackendCompilerFailed: backend='inductor' raised:
OutOfResources: out of resource: shared memory, Required: 115200, Hardware limit: 101376. Reducing block sizes or `num_stages` may help.

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True


In [ ]:
ms, min_ms, max_ms = triton.testing.do_bench(lambda: add(x, y), quantiles=quantiles)

In [6]:
hasattr(torch, "bfloat16")

True